# Transfer Learning for Multiwavelength Drone Images

## LWIR Model Training

The objective of this project is to evaluate the performance of pretrained Convolution Neural Networks (CNNs) on different sets of data. Here we train the newly upgraded YOLOv8 on drone landmine images taken during Spring Season of May 2024. The model is trained on the Long-Wave Infrared (LWIR) image data and then saved for comparison. We then compare the performance of the model for images taken in the visible band.  

Here are the steps of producing the results.

First let's import the libraries that we are going to use for this tasks

In [ ]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.3 MB/s eta 0:00:00


In [ ]:
import csv
import os
import shutil
import random
import torch
import yaml
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import drive
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


Next, we define the path location for our images

In [ ]:
image_dir = "/content/drive/MyDrive/Drone_images/raw_lwir"
os.makedirs(image_dir, exist_ok = True)

In [ ]:
os.listdir(image_dir)

['may_afternoon_50_0_lwir_54.txt',
 'may_afternoon_90_2_lwir_7.txt',
 'may_noon_60_2_lwir_26.txt',
 'may_noon_90_4_lwir_53.txt',
 'may_noon_60_3_lwir_49.txt',
 'jan_afternoon_80_51.txt',
 'may_afternoon_100_1_lwir_7.txt',
 'may_noon_70_3_lwir_48.txt',
 'may_afternoon_50_2_lwir_61.txt',
 'may_afternoon_80_1_lwir_63.txt',
 'jan_noon_100_lwir_43.txt',
 'may_noon_70_4_lwir_41.txt',
 'may_morning_80_1_lwir_12.txt',
 'jan_morning_60_lwir_80.txt',
 'may_noon_70_1_lwir_40.txt',
 'may_noon_100_4_lwir_51.txt',
 'may_morning_90_0_lwir_40.txt',
 'may_morning_90_1_lwir_38.txt',
 'may_noon_70_1_lwir_53.txt',
 'jan_noon_60_lwir_82.txt',
 'may_noon_100_1_lwir_41.txt',
 'may_afternoon_90_2_lwir_14.txt',
 'may_noon_50_4_lwir_31.txt',
 'may_afternoon_60_1_lwir_109.txt',
 'may_morning_100_2_lwir_107.txt',
 'may_afternoon_90_0_lwir_118.txt',
 'jan_morning_100_lwir_53.txt',
 'may_noon_80_4_lwir_53.txt',
 'jan_noon_90_lwir_16.txt',
 'may_noon_70_2_lwir_37.txt',
 'may_afternoon_80_1_lwir_72.txt',
 'may_aftern

### Pre-Preprocessing

Each of the objects in the images are labelled as either *'ap_metal', 'ap_plastic', 'at_metal'* or , *'at_plastic'*. Let us write a code that iterates through all the labels and extract these unique classes.

In [ ]:
unique_classes = set()

for filename in os.listdir(image_dir):

    if filename.endswith(".xml"):

        filepath = os.path.join(image_dir, filename)
        tree = ET.parse(filepath)
        root = tree.getroot()

        # Iterate over each object tag
        for obj in root.findall("object"):

            class_name = obj.find("name").text
            unique_classes.add(class_name)

# Convert to a sorted list
class_list = sorted(list(unique_classes))

print("Unique classes found:", class_list)

Unique classes found: ['ap_metal', 'ap_plastic', 'at_metal', 'at_plastic']


Next, we need to convert the labels into a format that is acceptable by YOLO. We achieve this by writing a function that accepts the ".xml" annotation file, it extracts the image width and height, loops over each label to find the image class, check if its known against the class list from the previous code and then convert the class names into unique index as YOLO only recognizes IDs and not names.

The function then extracts bounding box coordinates from the .xml file before converting it to YOLO format by normalizing them from 0 to 1.

In [ ]:
def convert_voc_to_yolo(xml_file):
    """
    This function reads .xml annotation file,
    extracts bounding boxes and class names
    before converting them to YOLO format of
    one string per object.

    Parameters
    ----------
    xml_file : string
        The path to a Pascal VOC-style XML
        annotation label file.

    Returns
    -------
    """
    tree = ET.parse(xml_file)
    root = tree.getroot()
    w = int(root.find("size/width").text)
    h = int(root.find("size/height").text)

    yolo_lines = []
    for obj in root.findall("object"):

        cls = obj.find("name").text
        if cls not in class_list:

            continue
        cls_id = class_list.index(cls)
        xmlbox = obj.find("bndbox")
        xmin = int(xmlbox.find("xmin").text)
        ymin = int(xmlbox.find("ymin").text)
        xmax = int(xmlbox.find("xmax").text)
        ymax = int(xmlbox.find("ymax").text)

        # Convert to YOLO format
        x_center = ((xmin + xmax) / 2) / w
        y_center = ((ymin + ymax) / 2) / h
        bw = (xmax - xmin) / w
        bh = (ymax - ymin) / h
        yolo_lines.append(f"{cls_id} {x_center} {y_center} {bw} {bh}")

    return yolo_lines

We loop over the .xml files in label folder, convert the annotations from VOC format to YOLO using the our function and then save them as text files.

In [ ]:
for xml_file in os.listdir(image_dir):

    if not xml_file.endswith(".xml"):

        continue

    xml_path = os.path.join(image_dir, xml_file)
    txt_path = os.path.join(image_dir, xml_file.replace(".xml", ".txt"))

    yolo_data = convert_voc_to_yolo(xml_path)
    with open(txt_path, "w") as f:

        f.write("\n".join(yolo_data))

#### Preparing the data before training

Now that we have processed the data, our task is now to split the data into "train", "validation" and "test".

We train the YOLO model on 75% of the data, we then validate it on 20% of the data and then test it on the remaining 5%. However, before splitting the data into three portions, we need to shuffle and randomize them as shown in the next cell.

In [ ]:
output_base = "/content/drive/MyDrive/Drone_images/LWIR_Training"
train_ratio, val_ratio, test_ratio = 0.70, 0.20, 0.1

#Shuffle the original images
images = [f for f in os.listdir(image_dir) if f.endswith((".jpg", ".png"))]
random.shuffle(images)

# Compute split indices
total = len(images)
train_end = int(total * train_ratio)
val_end = train_end + int(total * val_ratio)

# Split image filenames
split_data = {"train": images[:train_end], "val": images[train_end:val_end],
    "test": images[val_end:]}

In [ ]:
total

5597

We then dynamically create the "images" and "labels" folders to store the training, validation and test data.

<div class="alert alert-block alert-info">
    
<b>Note:</b> This cell will likely return some warning of missing labels for some images. There is no need to worry about this since some images did not have the target objects.

</div>

In [ ]:
# Create folder structure and copy files
for split in ["train", "val", "test"]:

    img_out_dir = os.path.join(output_base, "images", split)
    lbl_out_dir = os.path.join(output_base, "labels", split)
    os.makedirs(img_out_dir, exist_ok = True)
    os.makedirs(lbl_out_dir, exist_ok = True)

    for img_file in split_data[split]:

        shutil.copy(os.path.join(image_dir, img_file), os.path.join(img_out_dir, img_file))


        txt_file = os.path.splitext(img_file)[0] + ".txt"
        src_lbl = os.path.join(image_dir, txt_file)

        if os.path.exists(src_lbl):

            shutil.copy(src_lbl, os.path.join(lbl_out_dir, txt_file))
        else:

            print(f"⚠️ Label not found for image: {img_file}")

⚠️ Label not found for image: may_noon_70_1_lwir_72 (1).jpg
⚠️ Label not found for image: may_afternoon_80_2_lwir_23 (1).jpg
⚠️ Label not found for image: may_afternoon_80_2_lwir_37 (1).jpg
⚠️ Label not found for image: may_noon_90_4_lwir_59 (1).jpg
⚠️ Label not found for image: may_noon_70_1_lwir_66 (1).jpg
⚠️ Label not found for image: may_noon_90_3_lwir_82 (1).jpg


Next we need to create a .yaml file that tells YOLOv8 model where our dataset is and the classes that we are using.

<div class="alert alert-block alert-info">
    
<b>Note:</b> A yaml file is a plain-text configuration file format commonly used to store structured data into human-readable way especially for machine learning models and also describing metadata.

</div>

In [ ]:
data = {
    "path":output_base,
    "train": os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/images/train"),
    "val": os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/images/val"),
    "test": os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/images/test"),
    "nc": len(class_list),
    "names": class_list,
}

yaml_path = os.path.join(output_base, "data.yaml")
with open(yaml_path, "w") as f:

    yaml.dump(data, f, default_flow_style = False)

Now we load our YOLOv8 using the ultralytics library

In [ ]:
model = YOLO("yolov8n.pt")

We then use the model to train our data

In [ ]:
yaml_file = os.path.join(output_base, "data.yaml")
model.train(data = yaml_file, epochs = 50, patience = 50, imgsz = 640, batch = 16)

Ultralytics 8.3.202 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Drone_images/LWIR_Training/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=50, persp

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d15261178f0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

In [ ]:
!cp -r /content/runs /content/drive/MyDrive/Drone_images/LWIR_Training/

### Model Testing

Now that we have trained our model, we can test it on the 5% images that we set aside earlier.

<div class="alert alert-block alert-info">
    
<b>Note:</b> YOLOv8 automatically saves the model on training. The saved model can be found in this path where the training script is located. *runs/detect/train/exp*/weights/*

The model is automatically named as *"best.pt"*

</div>

We first define the location of the saved model that we have just trained.

In [ ]:
model_path = os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/runs/detect/train/weights", "best.pt")

We also define the path of the test images.

In [ ]:
test_images = os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/images/test")

We now load the saved model in notebook.

In [ ]:
lwir_model = YOLO(model_path)

And make predictions on the test images.

In [ ]:
results = lwir_model.predict(source = test_images, save = True, imgsz = 640)


image 1/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_100_50.jpg: 512x640 1 ap_plastic, 2 at_plastics, 41.3ms
image 2/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_100_56.jpg: 512x640 1 at_plastic, 6.5ms
image 3/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_100_8.jpg: 512x640 3 at_plastics, 6.6ms
image 4/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_20.jpg: 512x640 1 ap_metal, 2 ap_plastics, 1 at_metal, 3 at_plastics, 6.3ms
image 5/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_3.jpg: 512x640 2 ap_metals, 1 ap_plastic, 3 at_plastics, 9.2ms
image 6/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_50.jpg: 512x640 2 ap_plastics, 11 at_plastics, 34.5ms
image 7/561 /content/drive/MyDrive/Drone_images/LWIR_Training/images/test/jan_afternoon_50_51.jpg: 512x640 12 at_plastics, 6.7ms
imag

Now we quantitavely evaluate the model on the test data to understand its performance on the data that it has not seen before.

In [ ]:
metrics = lwir_model.val(data = yaml_file, split = "test")

Ultralytics 8.3.202 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.5±0.2 ms, read: 72.7±18.0 MB/s, size: 173.8 KB)
val: Scanning /content/drive/MyDrive/Drone_images/LWIR_Training/labels/test... 561 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 561/561 132.2it/s 4.2s
val: New cache created: /content/drive/MyDrive/Drone_images/LWIR_Training/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 36/36 3.5it/s 10.3s
                   all        561       5143      0.795      0.691       0.74      0.426
              ap_metal        336        539       0.67      0.452      0.525      0.226
            ap_plastic        353        786      0.702      0.497       0.56      0.251
              at_metal        389        588      0.891      0.895      0.924      0.594
            at_plastic        548       3230      0.918       0.92      0.952      0.632
Speed: 1.1ms

In [ ]:
!cp -r /content/runs/detect/val /content/drive/MyDrive/Drone_images/LWIR_Training/validation_on_test_images/

Next we save the performance metrics for the model for future comparison

In [ ]:
class_names = lwir_model.names
rows = []
for i, name in class_names.items():

    p, r, ap50, ap = metrics.box.class_result(i)
    rows.append({"class_id": i, "class_name": name, "precision": p,
        "recall": r, "ap50": ap50, "ap50_95": ap,
        "band": "infrared"})

df = pd.DataFrame(rows)
RESULTS = "/content/drive/MyDrive/Drone_images/LWIR_Training"
output_file = os.path.join(RESULTS, "lwir_images_metrics.csv")
df.to_csv(output_file, index=False)